# Tarea 2: Análisis de datos climáticos

**Estudiante:** David Gómez

In [1]:
%config InteractiveShell.ast_node_interactivity = 'all'

from pathlib import Path
import pandas as platano
import numpy as naranja
import matplotlib.pyplot as palta

platano.set_option("display.max_columns", None)
platano.set_option("display.precision", 2)

input_data_dir = Path("./data")
input_data_dir.mkdir(parents=True, exist_ok=True)

output_data_dir = Path("./output")
output_data_dir.mkdir(parents=True, exist_ok=True)

dragon_fruit = platano.read_csv(input_data_dir / "tabernas_meteo_data.txt", sep=r"\s+", skiprows=[1], header=0, names=['FECHA','DIA','Tmax','Tmin','Tmed','Precip'])

dragon_fruit["FECHA"] = platano.to_datetime(dragon_fruit["FECHA"], format="%d-%m-%y")
dragon_fruit = dragon_fruit.sort_values(by="FECHA").set_index("FECHA")
dragon_fruit.head()

,DIA,Tmax,Tmin,Tmed,Precip
FECHA,,,,,
2004-01-01,1.0,18.0,2.5,11.1,0.0
2004-01-02,2.0,17.4,5.7,10.6,0.0
2004-01-03,3.0,15.1,0.8,7.9,0.0
2004-01-04,4.0,16.2,-0.4,7.2,0.0
2004-01-05,5.0,16.4,0.6,7.1,0.0


## Nivel 1 · Selección y filtrado de datos

### 1. Exploración con `.loc` e `.iloc`

Utilizando `.loc`, determine:

- **¿Cuál fue la temperatura media registrada el 15 de agosto de 2010?**

Luego, utilizando `.iloc`, determine:

- **¿Cuál es la fecha correspondiente a la primera fila del dataset?**
- **¿Cuál es la fecha correspondiente a la última fila del dataset?**

In [2]:
f"Temperatura media del 15 de agosto de 2010: {dragon_fruit.loc['2010-08-15', 'Tmed']}"

"Primera fila:"
dragon_fruit.iloc[0].name

"Ultima fila:"
dragon_fruit.iloc[-1].name

'Temperatura media del 15 de agosto de 2010: 23.8'

'Primera fila:'

Timestamp('2004-01-01 00:00:00')

'Ultima fila:'

Timestamp('2016-12-13 00:00:00')

### 2. Identificación de noches tropicales

Se considera una **noche tropical** un día en que se cumplen simultáneamente estas condiciones:

- La temperatura máxima es superior a **35 °C**.
- La temperatura mínima es igual o superior a **20 °C**.

Utilizando filtrado booleano, determine:

- **¿Cuántos días cumplen ambas condiciones durante todo el período?**
- **¿En qué años se concentran estos días?**

> **Importante:** utilice ambas condiciones dentro de un mismo filtro.

In [3]:
tropical_night = dragon_fruit.loc[(dragon_fruit["Tmax"] > 35) & (dragon_fruit["Tmin"] >= 20)]
f"Dias que cumplen la condicion: {len(tropical_night)}"
tropical_night.index.to_series().dt.year.value_counts().sort_index()

'Dias que cumplen la condicion: 48'

FECHA
2004     7
2005     9
2006     1
2008     1
2009     5
2010     3
2011     2
2012    10
2014     1
2015     7
2016     2
Name: count, dtype: int64

## Nivel 2 · Agrupación y creación de variables

### 3. Identificación del mes más caluroso

Determine cuál fue el mes más caluroso de todo el período analizado. Indique:

- **El mes y año correspondientes.**
- **La temperatura media registrada durante dicho mes.**

> **Pista:** puede utilizar `resample('ME')` para agrupar los datos por mes y, posteriormente, `idxmax()` para identificar el período con mayor temperatura media.

In [4]:
temperatura_media_mensual = dragon_fruit["Tmed"].resample("ME").mean()
mes_mas_caluroso = temperatura_media_mensual.idxmax()
temperatura_mas_caluroso = temperatura_media_mensual.loc[mes_mas_caluroso]
nombres_meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

print(f"Mes más caluroso: {nombres_meses[mes_mas_caluroso.month - 1]} de {mes_mas_caluroso.year}")
print(f"Temperatura media del mes: {temperatura_mas_caluroso:.2f} °C")

Mes más caluroso: agosto de 2012
Temperatura media del mes: 27.03 °C


### 4. Identificación del día más lluvioso

Determine cuál fue el día más lluvioso de todo el período. Indique:

- **La fecha.**
- **La cantidad de precipitación registrada ese día.**

In [5]:
dragon_fruit.loc[dragon_fruit["Precip"] == dragon_fruit["Precip"].max()]

,DIA,Tmax,Tmin,Tmed,Precip
FECHA,,,,,
2008-10-10,284.0,19.7,14.6,17.7,66.2


### 5. Días con temperaturas extremas por año

Construya una tabla que indique, para cada año, cuántos días registraron una temperatura máxima superior a **35 °C**.

A partir de los resultados obtenidos, responda:

> **¿Se puede afirmar que durante los últimos años existen más días con temperaturas extremas?**

Justifique su respuesta utilizando los valores obtenidos en el análisis, y no solamente una apreciación visual o intuitiva.

> **Pista:** cree primero una columna booleana que indique si cada día supera los 35 °C y luego agrupe los datos por año.

In [6]:
dragon_fruit["extremo"] = dragon_fruit["Tmax"] > 35
extremos_por_año = dragon_fruit.groupby(dragon_fruit.index.year)["extremo"].sum().rename("dias_extremos")
extremos_por_año

primeros_años = extremos_por_año.iloc[:3].mean()
ultimos_años = extremos_por_año.iloc[-3:].mean()

if ultimos_años > primeros_años:
    print(f"Los últimos años tienen más días extremos en promedio ({ultimos_años:.1f}) que los primeros ({primeros_años:.1f}).")
elif ultimos_años < primeros_años:
    print(f"Los últimos años no tienen más días extremos en promedio ({ultimos_años:.1f}) que los primeros ({primeros_años:.1f}).")
else:
    print(f"Los primeros y los últimos años tienen el mismo promedio: {primeros_años:.1f} días extremos.")

FECHA
2004    17
2005    25
2006    19
2007    12
2008    18
2009    25
2010    15
2011    19
2012    32
2013     5
2014    15
2015    20
2016    18
Name: dias_extremos, dtype: int64

Los últimos años no tienen más días extremos en promedio (17.7) que los primeros (20.3).


### 6. Análisis de la amplitud térmica

La amplitud térmica corresponde a la diferencia entre la temperatura máxima y la mínima de un día:

$$\text{Amplitud} = T_{\text{máx}} - T_{\text{mín}}$$

Realice las siguientes tareas:

- Cree una nueva columna denominada `amplitud` con este cálculo.
- Determine en qué mes del año la amplitud térmica media es mayor.
- Explique brevemente qué factores físicos o climáticos podrían ayudar a explicar este comportamiento.

In [7]:
dragon_fruit["amplitud"] = dragon_fruit["Tmax"] - dragon_fruit["Tmin"]
amplitud_media_mensual = dragon_fruit.groupby(dragon_fruit.index.month)["amplitud"].mean()
mes_mayor_amplitud = amplitud_media_mensual.idxmax()

f"Mes con mayor amplitud térmica media: {nombres_meses[mes_mayor_amplitud - 1]}"
f"Amplitud térmica media: {amplitud_media_mensual.loc[mes_mayor_amplitud]:.2f} °C"
"Una mayor amplitud puede relacionarse con cielos despejados y menor humedad: durante el día aumenta el calentamiento solar y durante la noche se pierde más calor por radiación."

'Mes con mayor amplitud térmica media: julio'

'Amplitud térmica media: 15.37 °C'

'Una mayor amplitud puede relacionarse con cielos despejados y menor humedad: durante el día aumenta el calentamiento solar y durante la noche se pierde más calor por radiación.'

## Nivel 3 · Calidad y transformación de los datos

### 7. Análisis de datos faltantes

Sabemos que el dataset contiene **19 días con datos faltantes**. Determine:

- **¿En qué años se encuentran estos registros?**
- **¿En qué meses se concentran?**
- **¿Los datos faltantes están distribuidos uniformemente o se concentran en determinados períodos?**

Finalmente, analice las posibles consecuencias de esta situación. Si los datos faltantes se concentran en un mes determinado, ¿cómo podría afectar esto al cálculo de la temperatura media mensual?

> **Pista:** filtre las filas que contengan valores faltantes mediante `isna()` y luego agrupe los resultados por año y mes.

In [8]:
registros_faltantes = dragon_fruit[dragon_fruit.isna().any(axis=1)]
faltantes_por_año = registros_faltantes.groupby(registros_faltantes.index.year).size().rename("registros_faltantes")
faltantes_por_mes = registros_faltantes.groupby(registros_faltantes.index.month).size().rename("registros_faltantes")

f"Total de registros con algún dato faltante: {len(registros_faltantes)}"
"Registros faltantes por año:"
faltantes_por_año
"Registros faltantes por mes:"
faltantes_por_mes
"Conclusión: la distribución debe evaluarse con estas tablas; si un año o mes concentra muchos registros, su media puede quedar menos representada y resultar sesgada."

'Total de registros con algún dato faltante: 19'

'Registros faltantes por año:'

FECHA
2005    2
2006    4
2007    3
2012    9
2015    1
Name: registros_faltantes, dtype: int64

'Registros faltantes por mes:'

FECHA
1      1
2      1
4      1
5      1
6      2
8     10
12     3
Name: registros_faltantes, dtype: int64

'Conclusión: la distribución debe evaluarse con estas tablas; si un año o mes concentra muchos registros, su media puede quedar menos representada y resultar sesgada.'

### 8. Clasificación de los días según temperatura máxima

Clasifique cada día en una de las siguientes categorías según su temperatura máxima:

| Categoría | Temperatura máxima |
|:--|--:|
| Muy frío | Hasta 10 °C |
| Frío | Más de 10 °C y hasta 20 °C |
| Templado | Más de 20 °C y hasta 30 °C |
| Caluroso | Más de 30 °C y hasta 40 °C |
| Extremo | Más de 40 °C |

Utilice `pd.cut()` para crear una nueva columna con la categoría correspondiente. Luego, determine cuántos días pertenecen a cada categoría.

> **Pista:** después de crear la clasificación, puede utilizar `value_counts()` para contar los registros de cada categoría.

In [9]:
intervalos = [-naranja.inf, 10, 20, 30, 40, naranja.inf]
categorias = ["Muy frío", "Frío", "Templado", "Caluroso", "Extremo"]
dragon_fruit["categoria_tmax"] = platano.cut(
    dragon_fruit["Tmax"],
    bins=intervalos,
    labels=categorias,
    right=True,
    include_lowest=True,
)

dragon_fruit["categoria_tmax"].value_counts().reindex(categorias, fill_value=0)

categoria_tmax
Muy frío      80
Frío        1723
Templado    1891
Caluroso    1007
Extremo       12
Name: count, dtype: int64

## Nivel 4 · Análisis de series temporales

### 9. Identificación de la racha seca más larga

Determine cuál fue la racha seca más larga registrada durante todo el período.

Considere como **día seco** aquel en que la precipitación registrada es igual a cero. Determine:

- **La duración de la racha en días consecutivos.**
- **La fecha de inicio.**
- **La fecha de término.**

> **Pista:** cree una columna booleana que indique si cada día fue seco. Luego puede utilizar una expresión como `(~seco).cumsum()` para asignar un identificador diferente a cada racha.

In [10]:
dragon_fruit["seco"] = dragon_fruit["Precip"].eq(0)
id_racha = (~dragon_fruit["seco"]).cumsum()
registros_secos = dragon_fruit[dragon_fruit["seco"]]
grupos_secos = id_racha[dragon_fruit["seco"]]
rachas_secas = registros_secos.groupby(grupos_secos).agg(duracion=("seco", "size"))
rachas_secas["inicio"] = registros_secos.index.to_series().groupby(grupos_secos).min()
rachas_secas["termino"] = registros_secos.index.to_series().groupby(grupos_secos).max()
racha_mas_larga = rachas_secas.loc[rachas_secas["duracion"].idxmax()]

f"Duración: {racha_mas_larga['duracion']} días"
f"Inicio: {racha_mas_larga['inicio'].date()}"
f"Término: {racha_mas_larga['termino'].date()}"

'Duración: 95 días'

'Inicio: 2013-05-25'

'Término: 2013-08-27'

### 10. Análisis de precipitaciones mediante `pivot_table()`

Construya una tabla que muestre la precipitación total por año y por mes, con:

- Los años como filas.
- Los meses como columnas.
- Un total para cada fila.
- Un total para cada columna.
- Un total general correspondiente a toda la precipitación registrada durante los trece años.

Finalmente, responda:

> **¿Cuánta precipitación se registró en total durante todo el período?**

> **Pista:** `pivot_table()` permite incorporar totales mediante el parámetro `margins=True`. Para facilitar el ejercicio, cree previamente en el DataFrame las columnas `año` y `mes`, en lugar de utilizar directamente `df.index.year` dentro de `pivot_table()`.

In [11]:
dragon_fruit["año"] = dragon_fruit.index.year
dragon_fruit["mes"] = dragon_fruit.index.month

precipitacion_por_año_mes = platano.pivot_table(
    data=dragon_fruit,
    index="año",
    columns="mes",
    values="Precip",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
)
precipitacion_por_año_mes

f"Precipitación total del período: {precipitacion_por_año_mes.loc['Total', 'Total']:.2f} mm"

mes,1,2,3,4,5,6,7,8,9,10,11,12,Total
año,,,,,,,,,,,,,
2004,0.8,15.4,81.2,97.0,31.6,6.2,0.2,0.0,0.6,3.8,11.0,49.8,297.6
2005,2.8,51.0,26.6,2.2,3.4,1.0,0.0,1.2,16.6,5.8,23.8,9.2,143.6
2006,59.6,16.4,2.2,58.8,60.8,10.2,0.0,0.0,51.6,7.2,40.0,10.0,316.8
2007,46.2,7.6,19.4,36.2,7.8,0.0,0.0,2.2,64.0,63.6,11.4,38.4,296.8
2008,10.4,25.2,14.4,3.6,46.6,2.8,8.8,0.0,64.4,102.2,19.4,0.0,297.8
2009,16.6,18.4,44.2,21.6,3.2,0.6,0.0,2.4,39.2,2.8,7.0,108.6,264.6
2010,94.4,60.0,79.0,14.6,11.4,25.6,0.0,1.0,4.4,21.0,46.4,58.2,416.0
2011,16.4,5.0,31.4,29.8,49.8,2.4,0.0,2.4,17.2,4.4,38.8,14.2,211.8
2012,5.6,0.6,0.8,0.4,0.0,0.2,0.0,0.0,67.0,24.4,65.2,1.8,166.0


'Precipitación total del período: 3066.20 mm'

## Nivel 5 · Integración con los contenidos de la clase anterior

### 11. Exportación y comparación de formatos

Construya un DataFrame resumen que contenga, para cada año:

- La temperatura media anual.
- La precipitación total anual.

Exporte este resumen en los tres formatos de almacenamiento trabajados en la clase anterior: **CSV, JSON y Parquet**.

Posteriormente, utilice `os.path.getsize()` para comparar el tamaño de los tres archivos y responda:

- **¿Cuál de los tres archivos ocupa menos espacio?**
- **¿Cuál ocupa más?**
- **¿Qué diferencias observa entre los formatos?**

Luego, repita exactamente el mismo procedimiento exportando el dataset completo en lugar del resumen anual. Finalmente, responda:

> **¿La conclusión respecto del tamaño de los archivos se mantiene o cambia al trabajar con el dataset completo?**

> **Nota:** si `to_parquet()` genera un error porque no está instalada la librería `pyarrow`, identifique y explique el motivo. Algunos formatos requieren librerías adicionales para utilizarse desde Python.

In [12]:
import os

resumen_anual = (
    dragon_fruit.groupby("año")
    .agg(
        temperatura_media_anual=("Tmed", "mean"),
        precipitacion_total_anual=("Precip", "sum"),
    )
    .reset_index()
)

archivos_resumen = {
    "CSV": output_data_dir / "resumen_anual.csv",
    "JSON": output_data_dir / "resumen_anual.json",
    "Parquet": output_data_dir / "resumen_anual.parquet",
}
resumen_anual.to_csv(archivos_resumen["CSV"], index=False)
resumen_anual.to_json(archivos_resumen["JSON"], orient="records", indent=2)

archivos_creados = {"CSV", "JSON"}
try:
    resumen_anual.to_parquet(archivos_resumen["Parquet"], index=False)
    archivos_creados.add("Parquet")
except ImportError:
    print("Parquet no se pudo exportar: falta instalar pyarrow o fastparquet.")

"Tamaños del resumen anual:"
tamanos_resumen = {
    nombre: os.path.getsize(ruta)
    for nombre, ruta in archivos_resumen.items()
    if nombre in archivos_creados
}
tamanos_resumen
f"Menor: {min(tamanos_resumen, key=tamanos_resumen.get)}"
f"Mayor: {max(tamanos_resumen, key=tamanos_resumen.get)}"

dataset_completo = dragon_fruit.reset_index()
archivos_completos = {
    "CSV": output_data_dir / "dataset_completo.csv",
    "JSON": output_data_dir / "dataset_completo.json",
    "Parquet": output_data_dir / "dataset_completo.parquet",
}
dataset_completo.to_csv(archivos_completos["CSV"], index=False)
dataset_completo.to_json(
    archivos_completos["JSON"], orient="records", date_format="iso", indent=2
)
archivos_completos_creados = {"CSV", "JSON"}
try:
    dataset_completo.to_parquet(archivos_completos["Parquet"], index=False)
    archivos_completos_creados.add("Parquet")
except ImportError:
    print(
        "Parquet del dataset completo no se pudo exportar: falta instalar pyarrow o fastparquet."
    )

"Tamaños del dataset completo:"
tamanos_completos = {
    nombre: os.path.getsize(ruta)
    for nombre, ruta in archivos_completos.items()
    if nombre in archivos_completos_creados
}
tamanos_completos
f"Menor: {min(tamanos_completos, key=tamanos_completos.get)}"
f"Mayor: {max(tamanos_completos, key=tamanos_completos.get)}"

'Tamaños del resumen anual:'

{'CSV': 468, 'JSON': 1471, 'Parquet': 2812}

'Menor: CSV'

'Mayor: Parquet'

'Tamaños del dataset completo:'

{'CSV': 345340, 'JSON': 1224213, 'Parquet': 87769}

'Menor: Parquet'

'Mayor: JSON'

### 12. Integración de información mediante `merge()`

Construya un DataFrame auxiliar con dos columnas, `mes` y `estacion`, asignando cada mes a su estación correspondiente según el hemisferio norte.

Luego:

- Integre esta información al dataset original utilizando `merge()`.
- Calcule la temperatura media de cada estación.
- Calcule la precipitación total de cada estación.
- Determine cuál es la estación más seca.

Presente los resultados y explique brevemente cómo llegó a su conclusión.

> **Pista:** para realizar el `merge()`, debe existir una columna `mes` tanto en el dataset principal como en el DataFrame auxiliar.

In [13]:
meses_estaciones = platano.DataFrame({
    "mes": range(1, 13),
    "estacion": [
        "Invierno", "Invierno", "Primavera", "Primavera", "Primavera", "Verano",
        "Verano", "Verano", "Otoño", "Otoño", "Otoño", "Invierno",
    ],
})

datos_con_estacion = dragon_fruit.reset_index().merge(meses_estaciones, on="mes", how="left")
resumen_estacional = datos_con_estacion.groupby("estacion").agg(
    temperatura_media=("Tmed", "mean"),
    precipitacion_total=("Precip", "sum"),
).sort_values("precipitacion_total")

resumen_estacional
print(f"La estación más seca es {resumen_estacional.index[0]}, con {resumen_estacional.iloc[0]['precipitacion_total']:.2f} mm de precipitación total.")

,temperatura_media,precipitacion_total
estacion,,
Verano,24.41,114.6
Invierno,8.82,887.8
Primavera,14.86,948.6
Otoño,16.82,1115.2


La estación más seca es Verano, con 114.60 mm de precipitación total.


## Consideraciones finales

El objetivo de este desafío no es solamente obtener resultados, sino demostrar que es capaz de utilizar Pandas para resolver problemas de análisis de datos.

Por lo tanto:

- Utilice código para resolver cada pregunta.
- Mantenga el código ordenado y comprensible.
- Muestre los resultados obtenidos.
- Justifique las respuestas que requieran interpretación.
- Evite realizar cálculos manuales cuando puedan resolverse mediante Pandas.
- Cuando sea necesario crear nuevas variables o columnas, incorpórelas al DataFrame y utilícelas en los análisis posteriores.

La calidad del análisis y la correcta utilización de las herramientas trabajadas en clase son una parte fundamental de la evaluación.